In [ ]:
import pandas as pd
import io
import requests

# Downloading a clean version of the IMDB sentiment dataset (CSV)
url = "https://raw.githubusercontent.com/Ankit152/IMDB-Sentiment-Analysis/master/IMDB-Dataset.csv"
s = requests.get(url).content
df = pd.read_csv(io.StringIO(s.decode('utf-8')))

# Let's see what we're working with
print(f"Dataset Loaded: {df.shape[0]} rows")
print(df.head(3))

Dataset Loaded: 50000 rows
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive


In [ ]:
df.shape

(50000, 2)

In [ ]:
df.rename(columns={'review':'text'}, inplace=True)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [ ]:
df['text']

,text
0,One of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...
2,I thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...
4,"Petter Mattei's ""Love in the Time of Money"" is..."
...,...
49995,I thought this movie did a down right good job...
49996,"Bad plot, bad dialogue, bad acting, idiotic di..."
49997,I am a Catholic taught in parochial elementary...
49998,I'm going to have to disagree with the previou...


In [ ]:
df_sample =  df.sample(30000,random_state=42)

In [ ]:
df_sample.shape

(30000, 2)

In [ ]:
df_sample

,text,sentiment
33553,I really liked this Summerslam due to the look...,1
9427,Not many television shows appeal to quite as m...,1
199,The film quickly gets to a major chase scene w...,0
12447,Jane Austen would definitely approve of this o...,1
39489,Expectations were somewhat high for me when I ...,0
...,...,...
28567,Although Casper van Dien and Michael Rooker ar...,0
25079,I liked this movie. I wasn't really sure what ...,1
18707,Yes non-Singaporean's can't see what's the big...,1
15200,"As far as films go, this is likable enough. En...",0


In [ ]:
df_sample['sentiment'].value_counts()

,count
sentiment,
1,15011
0,14989


In [ ]:
import pandas as pd
import io
import requests
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


X_train, X_test, y_train, y_test = train_test_split(df_sample['text'], df_sample['sentiment'], test_size=0.2)

# Vectorization
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_vec = vectorizer.fit_transform(X_train)

# Train the Classical ML Model
model = LogisticRegression()
model.fit(X_train_vec, y_train)

print("Training complete! The model now understands real-world sentiment patterns.")

Training complete! The model now understands real-world sentiment patterns.


In [ ]:
from sklearn.metrics import r2_score , mean_absolute_error , accuracy_score

y_pred = model.predict(vectorizer.transform(X_test))

print(f"R2 Score: {r2_score(y_test, y_pred)}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test, y_pred)}")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred)}")

R2 Score: 0.5264957316257836
Mean Absolute Error: 0.11833333333333333
Accuracy Score: 0.8816666666666667


In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import r2_score , mean_absolute_error , accuracy_score

# 1. Initialize and train the Extra Trees Classifier
et_model = ExtraTreesClassifier(random_state=42) # Added random_state for reproducibility
et_model.fit(X_train_vec, y_train)

# 2. Make predictions on the test data
y_pred_et = et_model.predict(vectorizer.transform(X_test))

# 3. Evaluate the model
print(f"Extra Trees R2 Score: {r2_score(y_test, y_pred_et)}")
print(f"Extra Trees Mean Absolute Error: {mean_absolute_error(y_test, y_pred_et)}")
print(f"Extra Trees Accuracy Score: {accuracy_score(y_test, y_pred_et)}")

Extra Trees R2 Score: 0.4584711747607553
Extra Trees Mean Absolute Error: 0.13533333333333333
Extra Trees Accuracy Score: 0.8646666666666667


In [ ]:
from sklearn.pipeline import Pipeline
import joblib

# Create a pipeline that first vectorizes the text and then applies the logistic regression model
logistic_regression_pipeline = Pipeline([
    ('tfidf', vectorizer),
    ('logistic_regression', model)
])

# Export the pipeline
joblib.dump(logistic_regression_pipeline, 'logistic_regression_pipeline.pkl')

print("Logistic Regression model pipeline exported successfully as 'logistic_regression_pipeline.pkl'")

Logistic Regression model pipeline exported successfully as 'logistic_regression_pipeline.pkl'


In [ ]:
import gradio as gr
import joblib

# Load the pre-trained pipeline
logistic_regression_pipeline = joblib.load('logistic_regression_pipeline_negative.pkl')

# Define a prediction function
def predict_sentiment(text):
    prediction = logistic_regression_pipeline.predict([text])
    return 'Positive' if prediction[0] == 1 else 'Negative'

# Create the Gradio interface
iface = gr.Interface(fn=predict_sentiment,
                     inputs=gr.Textbox(lines=5, placeholder="Enter your text here..."),
                     outputs="text",
                     title="Sentiment Analysis",
                     description="predicted sentiment (Positive/Negative).")

# Launch the interface
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7759b2c0224f546cf5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Improving Text Preprocessing with Negation Handling

To make the model more robust to sentiment nuances, especially around negation, we will modify the text transformation function. The new function will identify negation words (e.g., 'not', 'never') and append a `_NEG` suffix to the word immediately following them. This helps the model distinguish between phrases like 'good' and 'not good' by creating distinct features ('good' vs. 'good_NEG').

We also need to ensure that these negation words are not removed by the stopword filter, as they are crucial for sentiment analysis.

In [173]:
import pandas as pd

# Define custom negative sentences and their labels
custom_negative_data = {
    'text': [
       "I cannot tolerate this terrible behavior anymore.",
        "She should not trust such dishonest people.",
        "We must not ignore these serious problems.",
        "I will not support this awful decision.",
        "They have not completed the work properly.",
        "I hate this dirty and crowded place.",
        "You cannot expect success without effort.",
        "He should not speak so rudely to everyone.",
        "We must not waste our valuable time here.",
        "I will not forgive this horrible mistake.",
        "She has not apologized for her bad attitude.",
        "I hate waiting in long and boring meetings.",
        "They cannot understand how painful this situation is.",
        "You should not make fun of others.",
        "We must not repeat this terrible error again.",
        "I will not buy this useless product.",
        "He has not shown any respect to the team.",
        "I hate the way they treat innocent people.",
        "She cannot handle this stressful environment.",
        "They should not spread false rumors online.",
        "We must not allow such negative behavior.",
        "I will not attend this boring event.",
        "You have not followed the instructions correctly.",
        "I hate noisy neighbors late at night.",
        "He cannot accept criticism from anyone.",
        "They should not ignore customer complaints.",
        "We must not support unfair practices.",
        "I will not watch this terrible movie again.",
        "She has not finished the assignment on time.",
        "I hate the unpleasant smell in this room.",
        "You cannot solve problems by shouting.",
        "He should not waste money on useless things.",
        "We must not break the company rules.",
        "I will not tolerate this disrespectful attitude.",
        "They have not prepared for the presentation.",
        "I hate fake promises and lies.",
        "She cannot survive under so much pressure.",
        "You should not enter without permission.",
        "We must not forget our responsibilities.",
        "I will not recommend this poor service to anyone.",
        "He has not learned from his past mistakes.",
        "I hate this confusing and messy system.",
        "They cannot finish the task before the deadline.",
        "You should not behave so immaturely.",
        "We must not damage public property.",
        "I will not listen to such nonsense.",
        "She has not recovered from the bad experience yet.",
        "I hate dealing with arrogant people.",
        "He cannot understand basic instructions properly.",
        "They should not arrive late every day.",
        "We must not ignore the warning signs.",
        "I will not stay in this uncomfortable hotel.",
        "You have not answered my important questions.",
        "I hate cold and unfriendly behavior.",
        "She cannot work in such toxic conditions.",
        "They should not disturb others during work hours.",
        "We must not repeat these careless mistakes.",
        "I will not use this broken device anymore.",
        "He has not completed the project successfully.",
        "I hate the terrible traffic in this city.",
        "You cannot force people to agree with you.",
        "She should not be so careless with money.",
        "We must not encourage negative habits.",
        "I will not participate in this unfair competition.",
        "They have not solved the issue completely.",
        "I hate rude customer service representatives.",
        "He cannot focus in noisy environments.",
        "You should not ignore your health problems.",
        "We must not make decisions in anger.",
        "I will not forget this disappointing experience.",
        "She has not improved her poor communication skills.",
        "I hate wasting time on meaningless tasks.",
        "They cannot manage the situation effectively.",
        "You should not blame others for your mistakes.",
        "We must not create unnecessary conflicts.",
        "I will not return to this awful restaurant.",
        "He has not kept his promises honestly.",
        "I hate people who constantly complain.",
        "She cannot deal with criticism professionally.",
        "They should not leave the office so early.",
        "We must not neglect important responsibilities.",
        "I will not accept this unfair treatment.",
        "You have not considered all the risks carefully.",
        "I hate dark and depressing places.",
        "He cannot understand the seriousness of the problem.",
        "They should not waste resources carelessly.",
        "We must not trust unreliable people.",
        "I will not spend money on this garbage product.",
        "She has not behaved professionally at work.",
        "I hate poor internet connections during meetings.",
        "You cannot ignore reality forever.",
        "He should not lie to his friends.",
        "We must not support harmful activities.",
        "I will not continue this useless conversation.",
        "They have not achieved the expected results.",
        "I hate the terrible design of this application.",
        "She cannot complete the task without help.",
        "You should not interrupt people while talking.",
        "We must not tolerate bullying in schools.",
        "I will not trust this suspicious company again.",
        "He has not shown any improvement recently.",
        "I hate the constant negativity around me.",
        "They cannot provide quality service consistently."
    ]
}

# Dynamically set the length of the sentiment list based on the text list
custom_negative_data['sentiment'] = [0] * len(custom_negative_data['text'])

df_custom_negative = pd.DataFrame(custom_negative_data)
df_sample = pd.concat([df_sample, df_custom_negative], ignore_index=True)

print(f"New sentences added. df_sample now has {df_sample.shape[0]} rows.")
print(df_sample.tail())

New sentences added. df_sample now has 30143 rows.
                                                    text  sentiment
30138          We must not tolerate bullying in schools.          0
30139    I will not trust this suspicious company again.          0
30140         He has not shown any improvement recently.          0
30141          I hate the constant negativity around me.          0
30142  They cannot provide quality service consistently.          0


In [175]:
df_sample.drop('text_processed',axis=1,inplace=True)

In [176]:
df_sample.tail()

,text,sentiment
30138,We must not tolerate bullying in schools.,0
30139,I will not trust this suspicious company again.,0
30140,He has not shown any improvement recently.,0
30141,I hate the constant negativity around me.,0
30142,They cannot provide quality service consistently.,0


In [177]:
# Import necessary libraries for text processing
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string

# Ensure NLTK data is downloaded
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

In [178]:
# Initialize stemmer
ps = PorterStemmer()

In [179]:
# Customize stopwords: remove negation words from the default list
custom_stopwords = set(stopwords.words('english'))
# Updated negation_words to only include true negation markers. Sentiment words like 'hate' and 'miso'
# should be learned by the model as regular negative terms, not as negators of subsequent words.
negation_words = {"not", "no", "n't", "never", "none", "hardly", "barely", "scarcely", "fewer", "neither", "nor", "don't", "doesn't", "didn't", "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", "hadn't", "won't", "wouldn't", "can't", "couldn't", "shouldn't", "mightn't", "mustn't"}
# Ensure negation words are not removed as stopwords
custom_stopwords = custom_stopwords - negation_words

def transform_with_negation(text):
    text = text.lower()
    tokens = nltk.word_tokenize(text)

    processed_tokens = []
    negated = False
    for token in tokens:
        # Handle contractions like "don't" which `nltk.word_tokenize` splits into "do" and "n't"
        if token in negation_words:
            negated = True
        elif token.isalnum(): # Process alphanumeric tokens
            if token not in custom_stopwords:
                stemmed_token = ps.stem(token)
                if negated:
                    processed_tokens.append(stemmed_token + "_NEG")
                    negated = False  # Reset negation flag after one word for simplicity
                else:
                    processed_tokens.append(stemmed_token)
        elif token in string.punctuation: # Reset negation if punctuation is encountered
            negated = False

    return " ".join(processed_tokens)


In [180]:
# Apply the new transformation to the 'text' column of our sample DataFrame
df_sample['text_processed'] = df_sample['text'].apply(transform_with_negation)

print("Text preprocessing with updated negation handling complete.")

Text preprocessing with updated negation handling complete.


### Re-training the Logistic Regression Model with Enhanced Features

Now, we will re-split the data, re-vectorize using the new `text_processed` column, and re-train the Logistic Regression model. This will allow the model to learn from the features that now include negation context.

In [181]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score

# Use the newly processed text data
X = df_sample['text_processed']
y = df_sample['sentiment']

# Split data into training and testing sets
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(X, y, test_size=0.2, random_state=42)

# Re-vectorize the text data
vectorizer_new = TfidfVectorizer(max_features=5000, stop_words='english') # Re-initialize vectorizer for new data
X_train_vec_new = vectorizer_new.fit_transform(X_train_new)
X_test_vec_new = vectorizer_new.transform(X_test_new)

# Re-train the Logistic Regression model
model_new = LogisticRegression(random_state=42)
model_new.fit(X_train_vec_new, y_train_new)

print("Logistic Regression model re-trained with enhanced text features.")

# Evaluate the re-trained model
y_pred_new = model_new.predict(X_test_vec_new)

print(f"\n--- Evaluation of Re-trained Logistic Regression Model ---")
print(f"R2 Score: {r2_score(y_test_new, y_pred_new)}")
print(f"Mean Absolute Error: {mean_absolute_error(y_test_new, y_pred_new)}")
print(f"Accuracy Score: {accuracy_score(y_test_new, y_pred_new)}")

Logistic Regression model re-trained with enhanced text features.

--- Evaluation of Re-trained Logistic Regression Model ---
R2 Score: 0.5362307692307693
Mean Absolute Error: 0.11593962514513187
Accuracy Score: 0.8840603748548681


In [182]:
df_sample.shape

(30143, 3)

### Exporting the New Logistic Regression Model Pipeline

If the re-trained model shows improved performance, we can export this new pipeline, which now incorporates the enhanced text preprocessing.

In [ ]:
from sklearn.pipeline import Pipeline
import joblib

# Create a new pipeline with the updated vectorizer and model
logistic_regression_pipeline_new = Pipeline([
    ('tfidf', vectorizer_new),
    ('logistic_regression', model_new)
])


# Export the new pipeline
joblib.dump(logistic_regression_pipeline_new, 'logistic_regression_pipeline.pkl')

print("New Logistic Regression model pipeline with negation handling exported successfully as 'logistic_regression_pipeline_negation_handled.pkl'")

New Logistic Regression model pipeline with negation handling exported successfully as 'logistic_regression_pipeline_negation_handled.pkl'


In [ ]:
# df_sample.drop('text',axis=1,inplace=True) # Commented out to prevent dropping the 'text' column

In [183]:
import gradio as gr
import joblib

# Load the pre-trained pipeline
logistic_regression_pipeline = joblib.load('logistic_regression_pipeline.pkl')


# Define a prediction function
def predict_sentiment(text):
    prediction = logistic_regression_pipeline.predict([text])
    return 'Positive' if prediction[0] == 1 else 'Negative'

# Create the Gradio interface
iface = gr.Interface(fn=predict_sentiment,
                     inputs=gr.Textbox(lines=5, placeholder="Enter your text here..."),
                     outputs="text",
                     title="Sentiment Analysis",
                     description="predicted sentiment (Positive/Negative).")

# Launch the interface
iface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://346fd75f44f26ec1d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
df_sample.to_csv('tranformed.csv')